# LDSR-S2: sampling speed and quality diagnostics

Compare **DDIM** and native **DPM-Solver++ 2M** using the canonical **v1 checkpoint**.
Run the cells in order. In Colab, select **Runtime → Change runtime type → GPU** first.

The default is a short **8-image × 1-seed** sweep. For the README benchmark cohort,
set `NUM_IMAGES = 64` and `SEEDS = [42, 43, 44]` before running the remaining cells.
The full sweep makes 2,880 predictions and can take much longer on a Colab GPU.
Data downloads are about 10 MB for the short sweep or 80 MB for all 64 pairs,
plus the v1 checkpoint (about 676 MB) and Python dependencies.

This notebook contains the fixed sample IDs, byte ranges and SHA256 checksums;
it needs no private scripts, Google Drive files or full TACO archive download.
It generates `sampler_speed_quality.png`, RGB/NIR diagnostic plots, a visual
comparison, individual measurements, the README-compatible summary CSV, and provenance.

The quality sweep uses **histogram matching**, just like the README measurements.
The README's separate SEN2NEON showcase uses back-projection and a different scene;
the visual comparison here uses the first fixed SEN2NAIP pair. Timings vary with GPU,
software and load, so a new run should not be expected to reproduce the quoted 6.39× exactly.

## 1. Install

Use Python 3.12 or newer. In a local checkout this installs that checkout;
in Colab it installs the repository revision below. Change `REPO_REF` to `dev`
while trying unpublished changes, or to a commit hash for a repeatable source version.
Run installation before importing the model; restart the kernel if you already imported another version.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_REF = "main"
local_repo = next((p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "opensr_model/srmodel.py").is_file()
                   and (p / "setup.py").is_file()), None)
package = (str(local_repo) if local_repo else
           f"git+https://github.com/ESAOpenSR/opensr-model.git@{REPO_REF}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       package, "pandas>=2.2", "huggingface_hub>=0.24"])

## 2. Configure the experiment

Keep `SPECTRAL_CORRECTION = "histogram"` to match the published measurement protocol.
Changing it to `"back_projection"` or `"none"` runs a separately labeled experiment.
DDIM uses eta 0.95 and temperature 1.0; DPM-Solver++ uses its native deterministic
updates, eta 0 and temperature 1.0. Both still sample conditioning and initial noise.
Seeds are reset before each call, matching those initial draws. Schedules and
later random draws differ between samplers. This compares their actual protocols.

All runs use FP32, batch size one, and central 128² LR → 512² HR crops. The fixed
spatially distinct cohort is exploratory; it is **not guaranteed unseen during training**.
Outputs go into a new timestamped directory so reruns preserve earlier results.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import hashlib
import importlib.metadata
import inspect
import json
import platform
import random
import time
from datetime import datetime, timezone
from importlib.resources import files

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
import torch
from affine import Affine
from huggingface_hub import hf_hub_download
from IPython.display import display
from omegaconf import OmegaConf
from scipy.ndimage import binary_erosion
from skimage.metrics import structural_similarity
from tqdm.auto import tqdm
from opensr_model import SRLatentDiffusion

NUM_IMAGES = 8  # Use 64 for the full README cohort.
SEEDS = [42]  # Use [42, 43, 44] for the full experiment.
DEVICE = "cuda"  # Explicitly choose "cpu" only for a slow CPU experiment.
SPECTRAL_CORRECTION = "histogram"
DDIM_STEPS = [5, 10, 20, 100]
DPM_STEPS = [1, 2, 3, 4, 5, 8, 10, 15, 20, 30, 50]
BOOTSTRAP_RESAMPLES = 2000
WARMUPS_PER_VARIANT = 1
OUTPUT_ROOT = Path.cwd() / "sampling_diagnostics"
CACHE_DIR = OUTPUT_ROOT / "cache"
RUN_DIR = OUTPUT_ROOT / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab, or explicitly set DEVICE = 'cpu'.")
if not 1 <= NUM_IMAGES <= 64 or not SEEDS or len(set(SEEDS)) != len(SEEDS):
    raise ValueError("Choose 1–64 images and distinct seeds.")
if any(not isinstance(s, int) or not 0 <= s < 2**32 for s in SEEDS):
    raise ValueError("Seeds must be integers in [0, 2**32).")
if SPECTRAL_CORRECTION not in ("histogram", "back_projection", "none"):
    raise ValueError("Unsupported spectral correction.")
if WARMUPS_PER_VARIANT < 1 or BOOTSTRAP_RESAMPLES < 1:
    raise ValueError("Warmups and bootstrap resamples must be positive.")
device = torch.device(DEVICE)
torch.set_num_threads(3)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=False)

def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)

def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize(device)

print("Device:", torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU")
print("Results:", RUN_DIR)

## 3. Fixed data manifest and bounded downloads

These are the same 64 evaluation pairs used for the README step sweep, in the
same order. Seeds within an image do not count as independent images.
[Source: SEN2NAIPv2](https://huggingface.co/datasets/tacofoundation/SEN2NAIPv2),
revision `c370504201072fdb1dd388013ab8c0fc7d00a57e`.

Only the selected TIFF byte ranges are requested. Each download must return HTTP
206 with the exact range and match the stored checksum. The code refuses a server
response that would download the entire archive. Cached TIFFs are checked again on reuse.

In [ ]:
DATASET_URL = 'https://huggingface.co/datasets/tacofoundation/SEN2NAIPv2/resolve/c370504201072fdb1dd388013ab8c0fc7d00a57e/sen2naipv2-crosssensor.taco'

# Fixed, spatially distinct evaluation cohort; no local paths.
EVALUATION_PAIRS = [
  {
    "sample_id": "NA5120_E1705N0676__m_3909255_ne_15_060_20220620",
    "spatial_group": "NA5120_E1705N0676",
    "lr": {
      "source_offset": 464706959,
      "bytes": 80432,
      "sha256": "c206c435954d9febc362469ced9783c8df98b1f5cccd95608ed47f7dcf993a4b"
    },
    "hr": {
      "source_offset": 464787391,
      "bytes": 1109696,
      "sha256": "5b33c9b054cd616d327c8b6cbd2cf925d25cce7b9e915911cbf59d09dfd4efb8"
    }
  },
  {
    "sample_id": "NA5120_E1196N0713__m_3712101_nw_10_060_20220519",
    "spatial_group": "NA5120_E1196N0713",
    "lr": {
      "source_offset": 5240812560,
      "bytes": 74158,
      "sha256": "e233530be81f8c20ba5e70a670e1b4117cdcf2e299158bcd331dca38eec8670b"
    },
    "hr": {
      "source_offset": 5240886718,
      "bytes": 1057473,
      "sha256": "452e591388ad83be1c39d9c055e36e5c361a91e4907ef9f0156f9c92427e05d1"
    }
  },
  {
    "sample_id": "NA5120_E1787N0539__m_3208831_nw_16_030_20211113",
    "spatial_group": "NA5120_E1787N0539",
    "lr": {
      "source_offset": 5255289844,
      "bytes": 75979,
      "sha256": "0070981ceb70205007d2ebf43ed90f17e32f5572bbeb3bdda3b89b9136f3aaa9"
    },
    "hr": {
      "source_offset": 5255365823,
      "bytes": 1153558,
      "sha256": "c232601ff6395152869bd28b4799cbbd18bce67b948ba9528f73cb4f786ba7dd"
    }
  },
  {
    "sample_id": "NA5120_E1913N0800__m_4307845_nw_17_060_20221107",
    "spatial_group": "NA5120_E1913N0800",
    "lr": {
      "source_offset": 33731864,
      "bytes": 76795,
      "sha256": "504427656146dd8269e40f41a8941ecd3dbc78af2ff4421807d6eaf656f13240"
    },
    "hr": {
      "source_offset": 33808659,
      "bytes": 1083956,
      "sha256": "c0895d512efb012777274e6efcbf7db08aa3b0094c7b994a67211838aa2e97e1"
    }
  },
  {
    "sample_id": "NA5120_E1250N0845__m_4412139_nw_10_060_20200625",
    "spatial_group": "NA5120_E1250N0845",
    "lr": {
      "source_offset": 6027896036,
      "bytes": 72115,
      "sha256": "89488879bb2e0dd8546a667710c542856ddc134611efc2a429f81aee6bb81856"
    },
    "hr": {
      "source_offset": 6027968151,
      "bytes": 1092551,
      "sha256": "9aab4aa3bdada7fa7149a03fb49c137596d6ae24ede0275b1e207696bdad1cd2"
    }
  },
  {
    "sample_id": "NA5120_E1791N0741__m_4108612_sw_16_060_20220616",
    "spatial_group": "NA5120_E1791N0741",
    "lr": {
      "source_offset": 400555387,
      "bytes": 78795,
      "sha256": "76cd7466820dd5900222cfb49c953d9bf40ea74daa6b1258c1e9c9cd53a83e0a"
    },
    "hr": {
      "source_offset": 400634182,
      "bytes": 1029214,
      "sha256": "65a878a899f720de868819262b6f1256cc47caa2357e505601addbea82655fd9"
    }
  },
  {
    "sample_id": "NA5120_E1846N0492__m_3008561_ne_16_060_20220129",
    "spatial_group": "NA5120_E1846N0492",
    "lr": {
      "source_offset": 4316426665,
      "bytes": 76446,
      "sha256": "86b7d10e25b23c672500a5e2402dacbba19b0212c4ca56cee703f2d40e827a35"
    },
    "hr": {
      "source_offset": 4316503111,
      "bytes": 1114428,
      "sha256": "ba5901ab536badd233329820451bd35419a050af22e3538cc1edb3fb7f13097e"
    }
  },
  {
    "sample_id": "NA5120_E1273N0833__m_4411944_sw_11_030_20220717",
    "spatial_group": "NA5120_E1273N0833",
    "lr": {
      "source_offset": 4087490652,
      "bytes": 76854,
      "sha256": "fb71da21d01a31606c11be8d8fa3774e07cc3d8662d8b37246d85f444abb8568"
    },
    "hr": {
      "source_offset": 4087567506,
      "bytes": 1106130,
      "sha256": "35cfa953d6d0c0a5d0b0446d6a496af43072d7d230cb7fa1803d09e45bcfabcd"
    }
  },
  {
    "sample_id": "NA5120_E1531N0703__m_4010237_nw_13_060_20210717",
    "spatial_group": "NA5120_E1531N0703",
    "lr": {
      "source_offset": 7666392842,
      "bytes": 67107,
      "sha256": "78c81a548fe8f87124cff8c9ab88625389cdafdcdbf85592cb80fea9bc5e6e6a"
    },
    "hr": {
      "source_offset": 7666459949,
      "bytes": 1017707,
      "sha256": "c9f00b064eeadab3d395362b05ceb92f2b8e9222fe9c49c6c1f266b4140c57f3"
    }
  },
  {
    "sample_id": "NA5120_E1377N0748__m_4111224_se_12_060_20210812",
    "spatial_group": "NA5120_E1377N0748",
    "lr": {
      "source_offset": 662062485,
      "bytes": 82537,
      "sha256": "70e588282bf7f2d8cf7a9e8aa99127d6872e96bc1c14949dc2a36ef14a7d0584"
    },
    "hr": {
      "source_offset": 662145022,
      "bytes": 1293598,
      "sha256": "bf6438a6d1d4a456422855e3af8003e85f8ad3731479ea806bea2c732dfbe554"
    }
  },
  {
    "sample_id": "NA5120_E1964N0745__m_4007646_se_18_060_20220720",
    "spatial_group": "NA5120_E1964N0745",
    "lr": {
      "source_offset": 2514014602,
      "bytes": 82092,
      "sha256": "72a818b9486606902b29ccb0bb3694b67c85a1ba0309a8e77cd52de2293ae5b5"
    },
    "hr": {
      "source_offset": 2514096694,
      "bytes": 1149884,
      "sha256": "83ace9eaca90d9aaa52feda19f68e451281ff117833a23c2373ddb7eb47e5ef7"
    }
  },
  {
    "sample_id": "NA5120_E1752N0703__m_4008947_sw_16_060_20190926",
    "spatial_group": "NA5120_E1752N0703",
    "lr": {
      "source_offset": 5502539915,
      "bytes": 77275,
      "sha256": "88e4d862f62439ff17c91fa7535e6e28b7da4a3445d114125ae99e827877584f"
    },
    "hr": {
      "source_offset": 5502617190,
      "bytes": 1081051,
      "sha256": "063835332627bcf644b610163592d1d304b2f7c78bcd573aedf0cb99671c1dc5"
    }
  },
  {
    "sample_id": "NA5120_E1580N0831__m_4609934_ne_14_060_20200818",
    "spatial_group": "NA5120_E1580N0831",
    "lr": {
      "source_offset": 488335074,
      "bytes": 73660,
      "sha256": "00043b1eda16a44788549cb3aca3c7b539382595affa805f6361e82d40c3dfec"
    },
    "hr": {
      "source_offset": 488408734,
      "bytes": 1060110,
      "sha256": "fa00123d65e55540a5dffee7e1c9b279bd611067d73feb2adfcc74ef7ce47fce"
    }
  },
  {
    "sample_id": "NA5120_E1210N0720__m_3812130_sw_10_060_20200617",
    "spatial_group": "NA5120_E1210N0720",
    "lr": {
      "source_offset": 655797306,
      "bytes": 82478,
      "sha256": "7408d9b887899055c3dfdd24f2138525c93e8b76ee7a99cf50cc28487d9c9c2d"
    },
    "hr": {
      "source_offset": 655879784,
      "bytes": 1177184,
      "sha256": "1d0869f80a3ad1cd56f7499bbe384c0b6450204518dc2f94f7e770d924e79c35"
    }
  },
  {
    "sample_id": "NA5120_E1562N0559__m_3310006_sw_14_060_20200930",
    "spatial_group": "NA5120_E1562N0559",
    "lr": {
      "source_offset": 3086459979,
      "bytes": 75976,
      "sha256": "1bdd77c327d0f3f9ff2eece9d801244dea0b78caf33c0a5b5388eced5d345548"
    },
    "hr": {
      "source_offset": 3086535955,
      "bytes": 1080756,
      "sha256": "2b7796f744d67c9cda12025aa198f928fb9f39e49191f85acc4dfadaeae6097f"
    }
  },
  {
    "sample_id": "NA5120_E1573N0597__m_3509918_ne_14_060_20190627",
    "spatial_group": "NA5120_E1573N0597",
    "lr": {
      "source_offset": 6228957875,
      "bytes": 74755,
      "sha256": "3769043d3896108a4db155d3b93da6ef35e5cc3088a74d52bd963a3bb71295f7"
    },
    "hr": {
      "source_offset": 6229032630,
      "bytes": 1080426,
      "sha256": "4f679d3a26fadab639215167155134545ee8e04c0e10d0e347916919b0244798"
    }
  },
  {
    "sample_id": "NA5120_E1491N0700__m_4010450_nw_13_060_20210726",
    "spatial_group": "NA5120_E1491N0700",
    "lr": {
      "source_offset": 3518202050,
      "bytes": 79229,
      "sha256": "fc7849db44ea41d14a28b970972f9aaf0e9357f2526ff8ea31de403cd66f533a"
    },
    "hr": {
      "source_offset": 3518281279,
      "bytes": 1121508,
      "sha256": "f56c41e974187c0987788b52ff7a8c7b05ca399f7ba0fcdebb4e3ebd87a11493"
    }
  },
  {
    "sample_id": "NA5120_E1252N0606__m_3411761_se_11_060_20200423",
    "spatial_group": "NA5120_E1252N0606",
    "lr": {
      "source_offset": 1247134972,
      "bytes": 85435,
      "sha256": "0c4691974f8225ae1970013dc7610d28d069f95f30109662d626b29500fe1ae9"
    },
    "hr": {
      "source_offset": 1247220407,
      "bytes": 1282435,
      "sha256": "1ece7a21b6274bc33e9e1fc4209203461f8eadb7a3b7f68f84ccf0e6f5bb1dc3"
    }
  },
  {
    "sample_id": "NA5120_E1975N0801__m_4207427_nw_18_060_20220415",
    "spatial_group": "NA5120_E1975N0801",
    "lr": {
      "source_offset": 8825864452,
      "bytes": 77020,
      "sha256": "2fc7c1c60e607d3932756860d55b4d8e7e48f26148c00f12bafb2f17ca5ca847"
    },
    "hr": {
      "source_offset": 8825941472,
      "bytes": 1101374,
      "sha256": "4ca8e44a91c6f144442910df393142bd7938b60db5cf676a7ea76f5e847d3967"
    }
  },
  {
    "sample_id": "NA5120_E1884N0602__m_3408212_se_17_060_20211017",
    "spatial_group": "NA5120_E1884N0602",
    "lr": {
      "source_offset": 3024212431,
      "bytes": 81413,
      "sha256": "745897167014495e75241c5463c86cb6e3f8e5061c8a1bad1357b8524427f4da"
    },
    "hr": {
      "source_offset": 3024293844,
      "bytes": 1165346,
      "sha256": "c23f58061caa9b0eed05b641a6affc5b4ee18843050fc213fff03184f8a3f284"
    }
  },
  {
    "sample_id": "NA5120_E1966N0726__m_3907637_nw_18_060_20211101",
    "spatial_group": "NA5120_E1966N0726",
    "lr": {
      "source_offset": 5201791629,
      "bytes": 79802,
      "sha256": "bc9e352e898f95faac92c0a26415dbbcaeb0606af9837422682a0ab7fb308a25"
    },
    "hr": {
      "source_offset": 5201871431,
      "bytes": 1059490,
      "sha256": "8351bc7c2951a3acb3c408a6ba3c6b99ba10f253c109347fc1e0bf5f5d293505"
    }
  },
  {
    "sample_id": "NA5120_E1204N0786__m_4112242_sw_10_060_20220715",
    "spatial_group": "NA5120_E1204N0786",
    "lr": {
      "source_offset": 3692955440,
      "bytes": 81719,
      "sha256": "361e1ae42bda0ae571b30f50769fd1071783a56acae9f5be28c8d2c8bed2ad4d"
    },
    "hr": {
      "source_offset": 3693037159,
      "bytes": 1232114,
      "sha256": "a36832cd8ec0bf39a6bdba520b4c1994d9cd81a5d16e5b76e6f0b89b56ebf54f"
    }
  },
  {
    "sample_id": "NA5120_E1222N0640__m_3511953_sw_11_060_20200521",
    "spatial_group": "NA5120_E1222N0640",
    "lr": {
      "source_offset": 2725795617,
      "bytes": 91529,
      "sha256": "eec6008ca8930c7f3deb105db12465c6361efad8aac24d61680349881bad950a"
    },
    "hr": {
      "source_offset": 2725887146,
      "bytes": 1329446,
      "sha256": "be1cab00cdf48b738ad39c96e88caf5f0518c44691f3ed481b38afa927480652"
    }
  },
  {
    "sample_id": "NA5120_E1666N0663__m_3809421_nw_15_060_20220629",
    "spatial_group": "NA5120_E1666N0663",
    "lr": {
      "source_offset": 7355780277,
      "bytes": 86198,
      "sha256": "5422cbbf938e35ecd18c98d23c8e931651642cd0475c14ca50887b2d38013b79"
    },
    "hr": {
      "source_offset": 7355866475,
      "bytes": 1214696,
      "sha256": "99bac75c2e6dda4d541b7df0004a5fe541b2a2a04bdd39329c58a3dee63f1647"
    }
  },
  {
    "sample_id": "NA5120_E1369N0549__m_3211027_ne_12_060_20211030",
    "spatial_group": "NA5120_E1369N0549",
    "lr": {
      "source_offset": 5486518063,
      "bytes": 76734,
      "sha256": "4e57cd909c3a91aa533c8a446fd700e8ed3f5af788ab5ab736dbd32e847a40a0"
    },
    "hr": {
      "source_offset": 5486594797,
      "bytes": 1179683,
      "sha256": "8527110fbcd2a11d7e1ec891c2879f897b66ff0ad885211cf6ec56542535ec59"
    }
  },
  {
    "sample_id": "NA5120_E1310N0816__m_4311708_ne_11_030_20220710",
    "spatial_group": "NA5120_E1310N0816",
    "lr": {
      "source_offset": 7422825842,
      "bytes": 80579,
      "sha256": "ccaaeeb0d8884c43e630060c137b3ba26c4327ff52f719dbcdf98c90a01931ed"
    },
    "hr": {
      "source_offset": 7422906421,
      "bytes": 1134315,
      "sha256": "29e96ef18e5aff1293fca0562f7340bacb3880ac1d0816ae04fb377268eb6df8"
    }
  },
  {
    "sample_id": "NA5120_E1438N0819__m_4510841_nw_12_060_20190821",
    "spatial_group": "NA5120_E1438N0819",
    "lr": {
      "source_offset": 8032079348,
      "bytes": 80017,
      "sha256": "00b613185ab917f42727754ad18bc7228df75dbe865bfb5fd41fd111d52214cf"
    },
    "hr": {
      "source_offset": 8032159365,
      "bytes": 1106916,
      "sha256": "429d397f54c5c91fe8e6bc8ec7503af7a83559a36fcaea2697b7aab73089191b"
    }
  },
  {
    "sample_id": "NA5120_E1991N0763__m_4007421_nw_18_060_20220704",
    "spatial_group": "NA5120_E1991N0763",
    "lr": {
      "source_offset": 7375011159,
      "bytes": 72992,
      "sha256": "e08720a8daaf24c237ad34f6a1cea0432ad76b2b0845e240017c43bd75dc313d"
    },
    "hr": {
      "source_offset": 7375084151,
      "bytes": 1022526,
      "sha256": "a61bc2d052bc59d212d490cb664cc72c36d1d0609603258e862ca40431f8fd8e"
    }
  },
  {
    "sample_id": "NA5120_E1561N0772__m_4310018_nw_14_060_20200813",
    "spatial_group": "NA5120_E1561N0772",
    "lr": {
      "source_offset": 8909126234,
      "bytes": 74901,
      "sha256": "a161f9735552aef0b33bed74967c2eb1cd52395576c2306aae534dd8e17504e9"
    },
    "hr": {
      "source_offset": 8909201135,
      "bytes": 1089434,
      "sha256": "57ef1d6225e864fbcec345f2080d43c0e2a49540fdeb9f3191669824862ef72a"
    }
  },
  {
    "sample_id": "NA5120_E1318N0551__m_3211354_nw_12_060_20211114",
    "spatial_group": "NA5120_E1318N0551",
    "lr": {
      "source_offset": 6060428613,
      "bytes": 80363,
      "sha256": "d3c6533796d2527fa10ee004c5c7ba20b7138461d83e16cb47b589320e7afd2d"
    },
    "hr": {
      "source_offset": 6060508976,
      "bytes": 1195849,
      "sha256": "eb17f4254c88f6958e4e18107851f4d796f94213b4b0f5266e4ad23812f7d511"
    }
  },
  {
    "sample_id": "NA5120_E1806N0632__m_3608620_nw_16_060_20220620",
    "spatial_group": "NA5120_E1806N0632",
    "lr": {
      "source_offset": 1131653983,
      "bytes": 88534,
      "sha256": "89923b73904ae750deb52c4b07f840fc92df167bb839690418d958eb7854ecfd"
    },
    "hr": {
      "source_offset": 1131742517,
      "bytes": 1275911,
      "sha256": "28069b9121e2a32f8c18e2dea202c5ec8c741f77a6c1bf877ae19685fd95b31a"
    }
  },
  {
    "sample_id": "NA5120_E1568N0841__m_4610004_nw_14_060_20200728",
    "spatial_group": "NA5120_E1568N0841",
    "lr": {
      "source_offset": 392444537,
      "bytes": 68197,
      "sha256": "cd7acea0c5456c4064fade3fdab25ddf298f84712d264849c20ff85fe5e46b6f"
    },
    "hr": {
      "source_offset": 392512734,
      "bytes": 969178,
      "sha256": "8e3d2f5301e3c0f76068c7bd91a91e3746e4339398c6b7871158258de89da3dd"
    }
  },
  {
    "sample_id": "NA5120_E1546N0622__m_3610114_sw_14_060_20190621",
    "spatial_group": "NA5120_E1546N0622",
    "lr": {
      "source_offset": 1628345224,
      "bytes": 85500,
      "sha256": "f56033ceaeaf5f159d6151ba25ead91080d42fc5207bb9b4bddc94537c0bb897"
    },
    "hr": {
      "source_offset": 1628430724,
      "bytes": 1198630,
      "sha256": "e7215b4f4e099e141adbc4675b5bf6b9fb4e3d51d5ef26cf3438f1d607aeab5c"
    }
  },
  {
    "sample_id": "NA5120_E1407N0674__m_3810928_sw_12_060_20210921",
    "spatial_group": "NA5120_E1407N0674",
    "lr": {
      "source_offset": 4414414557,
      "bytes": 83873,
      "sha256": "43291f472da66f3bbe377e4a7546f538edad260dd70b15effd8818854e953201"
    },
    "hr": {
      "source_offset": 4414498430,
      "bytes": 1254272,
      "sha256": "67626f16eee5b1a2205db786889ba335b595ee9646cbc9d0775dbcc57ea02ac4"
    }
  },
  {
    "sample_id": "NA5120_E1557N0524__m_3210044_ne_14_060_20200925",
    "spatial_group": "NA5120_E1557N0524",
    "lr": {
      "source_offset": 1328174825,
      "bytes": 67429,
      "sha256": "621401e8f399e9f9edb3e365098020b08827521c978203aa28870f1f3be97c86"
    },
    "hr": {
      "source_offset": 1328242254,
      "bytes": 1017933,
      "sha256": "119fcb57318667a55870e1ed8bbfecdee7c9ffd6fd93bd1013524f382801e4f8"
    }
  },
  {
    "sample_id": "NA5120_E1830N0544__m_3208525_se_16_060_20191001",
    "spatial_group": "NA5120_E1830N0544",
    "lr": {
      "source_offset": 3968402512,
      "bytes": 79162,
      "sha256": "00146310978c1217c4c8363d868ca79188f609b4eb2de7a92b06e8d4e3291c72"
    },
    "hr": {
      "source_offset": 3968481674,
      "bytes": 1073431,
      "sha256": "3dc40e483084d9ddb0e903e9eae343aa83b1b64c3ee7170a4faff337c8fcfab4"
    }
  },
  {
    "sample_id": "NA5120_E1763N0741__m_4108806_ne_16_060_20210928",
    "spatial_group": "NA5120_E1763N0741",
    "lr": {
      "source_offset": 2946669525,
      "bytes": 80318,
      "sha256": "37441edd61465cc8a6451e026cf91d4e77534593a393638d1ed8bf2e01bb032e"
    },
    "hr": {
      "source_offset": 2946749843,
      "bytes": 1108472,
      "sha256": "b04931e823939837c669d3f0efa9a6b23064f8241ed18dbf9376e8f6b8a20b9e"
    }
  },
  {
    "sample_id": "NA5120_E1561N0802__m_4510049_se_14_060_20200716",
    "spatial_group": "NA5120_E1561N0802",
    "lr": {
      "source_offset": 9345444482,
      "bytes": 71917,
      "sha256": "06bc7ca79091612b6b0b158fe9a2becbede96917e693509f1e62da297b4baeda"
    },
    "hr": {
      "source_offset": 9345516399,
      "bytes": 971022,
      "sha256": "28e6bde2ca28571d5fb5df5ce0b2609e7a1ae34269d9425106947d434f845024"
    }
  },
  {
    "sample_id": "NA5120_E1761N0796__m_4408840_ne_16_060_20220622",
    "spatial_group": "NA5120_E1761N0796",
    "lr": {
      "source_offset": 3202289482,
      "bytes": 87518,
      "sha256": "77dccf7047e29d388729d71267dd54718bbf88808ccb086e7f8e046cc8306bf8"
    },
    "hr": {
      "source_offset": 3202377000,
      "bytes": 1256871,
      "sha256": "53d92121425035615e97c1aa582a15cbe1cb92667585a80203435cf3c1542be3"
    }
  },
  {
    "sample_id": "NA5120_E1549N0848__m_4710149_ne_14_060_20200627",
    "spatial_group": "NA5120_E1549N0848",
    "lr": {
      "source_offset": 1959962244,
      "bytes": 72010,
      "sha256": "be171159b7d86864aa6ea7eb339a48a11eebece5d50798c95108a41673a7c7ab"
    },
    "hr": {
      "source_offset": 1960034254,
      "bytes": 1012522,
      "sha256": "9ee46b4b0acd976462a90dc3c77e60028ab5e29c38b01fc3028fed9b268db99c"
    }
  },
  {
    "sample_id": "NA5120_E1789N0599__m_3508742_ne_16_060_20211213",
    "spatial_group": "NA5120_E1789N0599",
    "lr": {
      "source_offset": 8048531842,
      "bytes": 78139,
      "sha256": "939828f186076622d27edb0bbad06a9d9470dacb384cfc2827e5525ca00e79fc"
    },
    "hr": {
      "source_offset": 8048609981,
      "bytes": 1196108,
      "sha256": "eaf5e123fd9350d4f3ccaa865907fa05e9a0c3526fc2927e576e3affd8eaeeef"
    }
  },
  {
    "sample_id": "NA5120_E1445N0781__m_4310823_sw_12_060_20190812",
    "spatial_group": "NA5120_E1445N0781",
    "lr": {
      "source_offset": 5628485685,
      "bytes": 85684,
      "sha256": "8c2c9e6028361e7b8bed5ef104d7bdd101831c1d06c464187acaa8109f9fdb59"
    },
    "hr": {
      "source_offset": 5628571369,
      "bytes": 1235894,
      "sha256": "13dc4c601dbd0f3f2d03e0543e7f0a7dad126b415b09c87af8bab0eb367d05f8"
    }
  },
  {
    "sample_id": "NA5120_E1839N0707__m_3908408_se_16_060_20191012",
    "spatial_group": "NA5120_E1839N0707",
    "lr": {
      "source_offset": 6079012831,
      "bytes": 82100,
      "sha256": "427344fddd4e8be7de23e92298de8d2cc343d4693ec61621cad43c1831071a30"
    },
    "hr": {
      "source_offset": 6079094931,
      "bytes": 1114407,
      "sha256": "1a925c5bccbcf29eaf57bf18cc3e703ae8bbcf4b0dcf0d80e8f54ee6d1c12f6a"
    }
  },
  {
    "sample_id": "NA5120_E1461N0891__m_4810815_se_12_060_20190722",
    "spatial_group": "NA5120_E1461N0891",
    "lr": {
      "source_offset": 9008155124,
      "bytes": 76676,
      "sha256": "3f34aaf94d5a3ca0078a08cbc2640aa05791891ed4a12b63e502ee4fcc7c30eb"
    },
    "hr": {
      "source_offset": 9008231800,
      "bytes": 1072164,
      "sha256": "5a42592b183054e7787a114048dbae8e279dced822a8e3d00e822d7c08d3eab8"
    }
  },
  {
    "sample_id": "NA5120_E1651N0623__m_3609505_se_15_060_20210825",
    "spatial_group": "NA5120_E1651N0623",
    "lr": {
      "source_offset": 5683622375,
      "bytes": 71291,
      "sha256": "0ae39f3e6e1463a39ff9f1e34cd31b729010ffe67b6f3e67d98f2257506653bf"
    },
    "hr": {
      "source_offset": 5683693666,
      "bytes": 950760,
      "sha256": "6ec974456959ed2346f7e4b05143ac5e04952a6ef1bf9e6742971c3ce84ecc98"
    }
  },
  {
    "sample_id": "NA5120_E1335N0587__m_3411258_se_12_060_20211114",
    "spatial_group": "NA5120_E1335N0587",
    "lr": {
      "source_offset": 1450972283,
      "bytes": 87972,
      "sha256": "4dd27bd16bbf1d4a34f37f238e7402df3235678741253d4955c4a71a2c9a0d60"
    },
    "hr": {
      "source_offset": 1451060255,
      "bytes": 1317928,
      "sha256": "3486bf412005495783b9c6ef244f41e7677d85ed937d9b284d7d45d193e537a9"
    }
  },
  {
    "sample_id": "NA5120_E1451N0594__m_3510652_se_13_060_20220601",
    "spatial_group": "NA5120_E1451N0594",
    "lr": {
      "source_offset": 944470482,
      "bytes": 93664,
      "sha256": "6694ef00f2b302c5344d42b2e5a560ce7eef21211bd7a5b43ea5ef799f5b8c48"
    },
    "hr": {
      "source_offset": 944564146,
      "bytes": 1402587,
      "sha256": "f25af73bc868784dc380f9047888568515144b712ebc3477d1ffea5d97a70458"
    }
  },
  {
    "sample_id": "NA5120_E1605N0429__m_2709701_nw_14_060_20200501",
    "spatial_group": "NA5120_E1605N0429",
    "lr": {
      "source_offset": 6613178474,
      "bytes": 76081,
      "sha256": "cc276f81dfb3c16ff9a2981e2c7714a1e9fa53bda3df3468234a8b94925df2dd"
    },
    "hr": {
      "source_offset": 6613254555,
      "bytes": 1073148,
      "sha256": "2fabf763646658566f2462697989fa8d53d036cd32bd58f4e1ef8195090582c8"
    }
  },
  {
    "sample_id": "NA5120_E1533N0768__m_4310236_nw_13_060_20200722",
    "spatial_group": "NA5120_E1533N0768",
    "lr": {
      "source_offset": 6294893924,
      "bytes": 87395,
      "sha256": "7f05d8c6b262bcb9f085ea6d5a1954fe4967fd0a2f79d4fa81f03d6b321a28ba"
    },
    "hr": {
      "source_offset": 6294981319,
      "bytes": 1235836,
      "sha256": "5f79612e28d54f0cfb60612b0a1561e40005fd2a3a2b10f28da2dec76e0114a4"
    }
  },
  {
    "sample_id": "NA5120_E1977N0758__m_4007522_nw_18_060_20220429",
    "spatial_group": "NA5120_E1977N0758",
    "lr": {
      "source_offset": 6414020114,
      "bytes": 84430,
      "sha256": "9161408ecc6077c990cd20f183a08d0609d8e917647936a030dbf1a9ce653a9c"
    },
    "hr": {
      "source_offset": 6414104544,
      "bytes": 1132749,
      "sha256": "67309b1852421f8386cfa2747c9337c69f9ef13e1ea76d15bd6d75613b4b1f2b"
    }
  },
  {
    "sample_id": "NA5120_E1320N0870__m_4611730_se_11_060_20190630",
    "spatial_group": "NA5120_E1320N0870",
    "lr": {
      "source_offset": 7983623610,
      "bytes": 71925,
      "sha256": "59a5c3ce8f6fead33ca98c6ee5fc49d491ba323033dcda73c65a59bd67005dac"
    },
    "hr": {
      "source_offset": 7983695535,
      "bytes": 940679,
      "sha256": "d1cf0cc7a03df68a18eabf649ac9d4bd1d73d7b695335123cb8dec3d8a6a7f76"
    }
  },
  {
    "sample_id": "NA5120_E1808N0698__m_3908517_nw_16_060_20200611",
    "spatial_group": "NA5120_E1808N0698",
    "lr": {
      "source_offset": 6925695538,
      "bytes": 78852,
      "sha256": "5db1f9cadd51c31a8d17d40c704b68e7bf105f02045392a64e8ea0a52a022cf2"
    },
    "hr": {
      "source_offset": 6925774390,
      "bytes": 1059050,
      "sha256": "af27205ffa81d199430ab2ef72209fb3b19fc8c24cce712211eacb0ffa279476"
    }
  },
  {
    "sample_id": "NA5120_E1575N0630__m_3709951_nw_14_060_20190712",
    "spatial_group": "NA5120_E1575N0630",
    "lr": {
      "source_offset": 2761207822,
      "bytes": 72980,
      "sha256": "1b1e489e36fc5a970ca191da42fc6fc6a8126c15519339e696a0aff2bc489b8d"
    },
    "hr": {
      "source_offset": 2761280802,
      "bytes": 950760,
      "sha256": "b2fc3d3ecaf6bb84d23e2fd97a0ac628d2d4e960dcc52849b6498c0db9c7cb5f"
    }
  },
  {
    "sample_id": "NA5120_E1386N0634__m_3611029_ne_12_060_20211020",
    "spatial_group": "NA5120_E1386N0634",
    "lr": {
      "source_offset": 2719561147,
      "bytes": 81380,
      "sha256": "d3db636d68a33ad3b6dc505e1f06a0df587432c4db0c4a86f1b12f21f0211e4e"
    },
    "hr": {
      "source_offset": 2719642527,
      "bytes": 1206773,
      "sha256": "564daaa80e5de5cb40125b789c3f19db77c71702c4cb85598869d34f9b381fd5"
    }
  },
  {
    "sample_id": "NA5120_E1592N0395__m_2609836_sw_14_060_20220529",
    "spatial_group": "NA5120_E1592N0395",
    "lr": {
      "source_offset": 8803026700,
      "bytes": 81916,
      "sha256": "ea023474d652d70e911fedeea0e3f10de5a808dea6a23a0cbf7552bb9a7529f8"
    },
    "hr": {
      "source_offset": 8803108616,
      "bytes": 1219167,
      "sha256": "7f906136ed9b83466aab4d1957dede83bca79e3f77de6f8c9e7464ee554681c5"
    }
  },
  {
    "sample_id": "NA5120_E1438N0665__m_3810743_nw_13_060_20190913",
    "spatial_group": "NA5120_E1438N0665",
    "lr": {
      "source_offset": 5707077302,
      "bytes": 79711,
      "sha256": "a0fc3e2e680945e339d08e2a3799602ce7a30dcd814f3c563ed2231d952d1547"
    },
    "hr": {
      "source_offset": 5707157013,
      "bytes": 1212973,
      "sha256": "9c7a190431736f07244a82b227032e78a91263bed8946c55b7d24695c0b33ece"
    }
  },
  {
    "sample_id": "NA5120_E1615N0428__m_2709705_sw_14_060_20220516",
    "spatial_group": "NA5120_E1615N0428",
    "lr": {
      "source_offset": 9182551462,
      "bytes": 68955,
      "sha256": "a1a647a7b9fc70668391db9f4382180a3d5553c064db4898a1c8f47278ea31b8"
    },
    "hr": {
      "source_offset": 9182620417,
      "bytes": 1016707,
      "sha256": "99b47f9fba2748b402c82f323b52d269cbc2dad9d70f8b92d7c59b66b9436853"
    }
  },
  {
    "sample_id": "NA5120_E1410N0702__m_3910911_ne_12_060_20210922",
    "spatial_group": "NA5120_E1410N0702",
    "lr": {
      "source_offset": 4075952936,
      "bytes": 87502,
      "sha256": "bb88b075dfe2fc23d71cd54ee5bc89a7fff28573f57c1b663d41488e206dd952"
    },
    "hr": {
      "source_offset": 4076040438,
      "bytes": 1320780,
      "sha256": "08658be009667a07e1bf11796e913f15c13845d51ef09cc0ec5fa2e3a91b8655"
    }
  },
  {
    "sample_id": "NA5120_E1296N0855__m_4511828_se_11_060_20200721",
    "spatial_group": "NA5120_E1296N0855",
    "lr": {
      "source_offset": 3794816204,
      "bytes": 76960,
      "sha256": "122487b88f68589e56da832ad197fc14899f5f02f2c44be60796c4899c4f6b47"
    },
    "hr": {
      "source_offset": 3794893164,
      "bytes": 1174763,
      "sha256": "9c37d265d8e9e1709be0fd66a3d9d5f2ac35dfb9403891ac9f8f3164fb9d1cbf"
    }
  },
  {
    "sample_id": "NA5120_E1838N0795__m_4308307_se_17_060_20200702",
    "spatial_group": "NA5120_E1838N0795",
    "lr": {
      "source_offset": 1642994701,
      "bytes": 76887,
      "sha256": "33050f1bdba2fbaa4f914405ab46a1634ac645c5007623a686380fc709bb5f65"
    },
    "hr": {
      "source_offset": 1643071588,
      "bytes": 1038575,
      "sha256": "0930a917951820756a93995c796c8a46370a76037d178e23f81c9656d7dd5034"
    }
  },
  {
    "sample_id": "NA5120_E1613N0476__m_3009752_se_14_060_20220611",
    "spatial_group": "NA5120_E1613N0476",
    "lr": {
      "source_offset": 2571373145,
      "bytes": 76793,
      "sha256": "7b1182a302b948226ab3ee71c39b62a576a2006ce5180cb98b0e2de61b6b59a1"
    },
    "hr": {
      "source_offset": 2571449938,
      "bytes": 1195379,
      "sha256": "463a0510a955749670a84a6fcca652db43f16d7d705f7c3f6284471dc15200e5"
    }
  },
  {
    "sample_id": "NA5120_E1491N0656__m_3810452_nw_13_060_20190921",
    "spatial_group": "NA5120_E1491N0656",
    "lr": {
      "source_offset": 3013384532,
      "bytes": 75420,
      "sha256": "6d827e23e7a245c404bc3b419bdc95f2df0b4ca0a4a93222c286775132b05559"
    },
    "hr": {
      "source_offset": 3013459952,
      "bytes": 1146770,
      "sha256": "9be93d182592971e6d95c2d10df6ef4f13110ece1754af4029a5eeb2419b60e0"
    }
  },
  {
    "sample_id": "NA5120_E1798N0768__m_4208501_nw_16_060_20200702",
    "spatial_group": "NA5120_E1798N0768",
    "lr": {
      "source_offset": 353730647,
      "bytes": 82478,
      "sha256": "bd6e88002d008cd64cc28378e6abe08814addf16dfbbb79d5368c87d97ed4113"
    },
    "hr": {
      "source_offset": 353813125,
      "bytes": 1209411,
      "sha256": "76c34fb872f43b32e07d5aecba0de97a7a7551dc3cc036690272b06288466bd5"
    }
  },
  {
    "sample_id": "NA5120_E1280N0814__m_4311826_sw_11_060_20200725",
    "spatial_group": "NA5120_E1280N0814",
    "lr": {
      "source_offset": 6496587363,
      "bytes": 75354,
      "sha256": "375c0bfdc22610d0f57238eebd43aa24108de2882e4adc8e6a0ffc976c86f4b6"
    },
    "hr": {
      "source_offset": 6496662717,
      "bytes": 1180001,
      "sha256": "11bfe7c63732c25cb1d8191beb24c205d9033b1b5576174b11ba74794d9ec6a9"
    }
  }
]

In [ ]:
def download_asset(record, band):
    entry = record[band]
    destination = CACHE_DIR / record["sample_id"] / f"{band}.tif"
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and sha256(destination) == entry["sha256"]:
        return destination
    start, length = entry["source_offset"], entry["bytes"]
    if start < 0 or not 0 < length <= 32 * 1024 * 1024:
        raise ValueError("Unexpected asset range.")
    end = start + length - 1
    url = f"{DATASET_URL}?download=true&sampling_range={start}-{end}"
    for attempt in range(4):
        try:
            with requests.get(url, headers={"Range": f"bytes={start}-{end}",
                                           "Accept-Encoding": "identity"},
                              stream=True, timeout=(20, 120)) as response:
                response.raise_for_status()
                if response.status_code != 206:
                    raise RuntimeError("Server ignored Range; refusing full archive download.")
                if not response.headers.get("Content-Range", "").startswith(f"bytes {start}-{end}/"):
                    raise RuntimeError("Server returned an incorrect byte range.")
                blob = response.raw.read(length + 1)
            if len(blob) != length or hashlib.sha256(blob).hexdigest() != entry["sha256"]:
                raise RuntimeError("Asset length or checksum mismatch.")
            temporary = destination.with_suffix(".tmp")
            temporary.write_bytes(blob)
            temporary.replace(destination)
            return destination
        except (requests.RequestException, RuntimeError):
            if attempt == 3:
                raise
            time.sleep(2**attempt)

selected_pairs = EVALUATION_PAIRS[:NUM_IMAGES]
assert len({p["spatial_group"] for p in selected_pairs}) == NUM_IMAGES
for record in tqdm(selected_pairs, desc="Download/check paired TIFFs"):
    for band in ("lr", "hr"):
        download_asset(record, band)
(RUN_DIR / "data_manifest.json").write_text(json.dumps(
    {"source_url": DATASET_URL, "pairs": selected_pairs}, indent=2))

In [ ]:
def read_pair(record):
    with rasterio.open(CACHE_DIR / record["sample_id"] / "lr.tif") as low, rasterio.open(CACHE_DIR / record["sample_id"] / "hr.tif") as high:
        if low.count != 4 or high.count != 4:
            raise ValueError("Expected RGB+NIR bands")
        if any(dtype != "uint16" for dtype in low.dtypes + high.dtypes):
            raise ValueError("Expected UINT16 reflectance rasters")
        if low.nodata != 65535 or high.nodata != 65535:
            raise ValueError("Expected SEN2NAIP nodata 65535")
        if (high.width, high.height) != (low.width * 4, low.height * 4):
            raise ValueError("Expected aligned 4x LR/HR shapes")
        delta = np.asarray(low.transform) - np.asarray(high.transform * Affine.scale(4))
        if low.crs is None or low.crs != high.crs or not np.allclose(delta, 0, rtol=0, atol=0.001):
            raise ValueError("LR/HR georeferencing is not aligned")
        size = 128
        if min(low.width, low.height) < size:
            raise ValueError("Raster smaller than 128px crop")
        x, y = (low.width - size) // 2, (low.height - size) // 2
        window = rasterio.windows.Window(x, y, size, size)
        hr_window = rasterio.windows.Window(x * 4, y * 4, size * 4, size * 4)
        lr_raw, hr_raw = low.read(window=window), high.read(window=hr_window)
        lr_valid = (low.read_masks(window=window) > 0).all(0) & (lr_raw != 65535).all(0)
        hr_valid = (high.read_masks(window=hr_window) > 0).all(0) & (hr_raw != 65535).all(0)
        mask = hr_valid & np.repeat(np.repeat(lr_valid, 4, 0), 4, 1)
        if not mask.any():
            raise ValueError("No valid evaluation pixels")
        lr, hr = lr_raw.astype(np.float32) * 1e-4, hr_raw.astype(np.float32) * 1e-4
        meta = dict(valid_fraction=float(mask.mean()), lr_crop_origin=[x, y],
                    clipped_lr_fraction=float((lr[:, lr_valid] > 1).mean()),
                    clipped_hr_fraction=float((hr[:, hr_valid] > 1).mean()))
        lr = np.where(lr_valid[None], np.clip(lr, 0, 1), 0).astype(np.float32)
        hr = np.where(hr_valid[None], np.clip(hr, 0, 1), 0).astype(np.float32)
    return lr, hr, mask, meta


pairs = [read_pair(record) for record in selected_pairs]
print(f"Loaded {len(pairs)} aligned 4-band pairs.")

## 4. Load v1 and record the environment

The checkpoint is downloaded with the [Hugging Face cache](https://huggingface.co/docs/huggingface_hub/guides/download)
and verified against the SHA256 used by the original benchmark. The package's
native configuration and loader restore the checkpoint's schedule. The resolved
configuration, checkpoint hash, module hashes, dependency versions and device
are saved alongside the measurements.

In [ ]:
CHECKPOINT_NAME = "opensr-ldsrs2_v1_0_0.ckpt"
CHECKPOINT_SHA256 = "e2621e3912eb7c14867c3d20c9029607ba941be8e166dc09621860fcac27dc3a"
# Reuse a matching local checkpoint when working in the repository.
local_checkpoint = local_repo / CHECKPOINT_NAME if local_repo else None
checkpoint_path = (local_checkpoint if local_checkpoint and local_checkpoint.is_file() else
                   Path(hf_hub_download("simon-donike/RS-SR-LTDF", CHECKPOINT_NAME)))
if sha256(checkpoint_path) != CHECKPOINT_SHA256:
    raise RuntimeError("Checkpoint differs from the canonical v1 benchmark weights.")
config = OmegaConf.load(str(files("opensr_model").joinpath("configs/config_10m.yaml")))
seed_all(42)
model = SRLatentDiffusion(config, device=device)
model.load_pretrained(str(checkpoint_path))
model.eval()

variants = [dict(variant=f"ddim_s{s}", sampler="ddim", steps=s, eta=0.95)
            for s in DDIM_STEPS]
variants += [dict(variant=f"dpm_2m_s{s}", sampler="dpm_solver++", steps=s, eta=0.0)
             for s in DPM_STEPS]
if "ddim_s100" not in {v["variant"] for v in variants}:
    raise ValueError("The paired DDIM-100 baseline is required.")
if len({v["variant"] for v in variants}) != len(variants):
    raise ValueError("Duplicate sampler variants.")
for variant in variants:
    steps = variant["steps"]
    if not isinstance(steps, int) or not 1 <= steps < model.model.num_timesteps:
        raise ValueError("Invalid step count.")
    if variant["sampler"] == "ddim" and model.model.num_timesteps % steps:
        raise ValueError("DDIM step counts must divide the training timestep count.")

versions = {name: importlib.metadata.version(name) for name in
            ("opensr-model", "torch", "diffusers", "numpy", "scikit-image", "rasterio", "pandas")}
module_paths = [Path(inspect.getfile(SRLatentDiffusion)),
                Path(inspect.getfile(SRLatentDiffusion)).parent / "diffusion/dpm_solver.py",
                Path(inspect.getfile(SRLatentDiffusion)).parent / "diffusion/utils.py"]
provenance = dict(
    created_utc=datetime.now(timezone.utc).isoformat(), python=platform.python_version(),
    versions=versions, device=str(device),
    gpu=torch.cuda.get_device_name(device) if device.type == "cuda" else None,
    cuda=torch.version.cuda, precision="float32", tf32=False, cpu_threads=torch.get_num_threads(),
    checkpoint=CHECKPOINT_NAME, checkpoint_sha256=CHECKPOINT_SHA256,
    dataset_url=DATASET_URL, images=NUM_IMAGES, seeds=SEEDS, variants=variants,
    spectral_correction=SPECTRAL_CORRECTION, bootstrap_resamples=BOOTSTRAP_RESAMPLES,
    warmups_per_variant=WARMUPS_PER_VARIANT,
    effective_model_config=OmegaConf.to_container(model.config, resolve=True),
    source_sha256={p.name: sha256(p) for p in module_paths},
)
if local_repo:
    provenance["git_commit"] = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=local_repo, text=True).strip()
    provenance["git_dirty"] = bool(subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=local_repo, text=True).strip())
else:
    distribution = importlib.metadata.distribution("opensr-model")
    provenance["package_source"] = json.loads(distribution.read_text("direct_url.json") or "{}")
(RUN_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2))
print(versions)
print(f"{len(variants)} variants × {NUM_IMAGES} images × {len(SEEDS)} seeds")

## 5. Metrics and timed inference

Timing includes the complete model forward pass (encoding, schedule creation,
denoising, decoding and correction), with CUDA synchronization before and after.
It excludes downloads, loading, explicit input transfers, CPU copies, metrics and
file writes. Every variant is warmed up first and execution order is shuffled
for each image/seed. A hook verifies the actual number of denoiser evaluations.

PSNR uses reflectance `data_range=1`; SSIM uses Gaussian weights, sigma 1.5,
and population covariance. Scores use the original valid pixels; SSIM support
is eroded by an 11×11 window so invalid pixels cannot affect it. LR and HR inputs
are clipped to [0, 1] after scaling by 1e-4; **predictions are not clipped** for metrics.

In [ ]:
def image_metrics(prediction: np.ndarray, target: np.ndarray, mask: np.ndarray) -> dict:
    if prediction.shape != target.shape or prediction.ndim != 3 or mask.shape != target.shape[1:]:
        raise ValueError("Expected matching CHW images and an HW validity mask")
    if not np.isfinite(prediction).all() or not np.isfinite(target).all():
        raise FloatingPointError("Non-finite image passed to metrics")
    if not mask.any():
        raise ValueError("No valid pixels")
    prediction, target = prediction.astype(np.float64), target.astype(np.float64)
    result = {"valid_fraction": float(mask.mean())}
    support = binary_erosion(mask, structure=np.ones((11, 11)), border_value=0)
    for name, channels in (("all", slice(None)), ("rgb", slice(0, 3)), ("nir", slice(3, 4))):
        p, t = prediction[channels], target[channels]
        error = (p - t)[:, mask]
        mse = float(np.mean(error ** 2))
        result.update({f"{name}_mae": float(np.abs(error).mean()),
                       f"{name}_rmse": float(np.sqrt(mse)),
                       # Exact equality is encoded without non-standard JSON Infinity.
                       f"{name}_psnr": None if mse == 0 else float(-10 * np.log10(mse)),
                       f"{name}_exact_match": mse == 0,
                       f"{name}_out_of_range": float(((p < 0) | (p > 1))[:, mask].mean())})
        maps = []
        if support.any():
            for a, b in zip(p, t):
                _, spatial = structural_similarity(a, b, data_range=1.0, gaussian_weights=True,
                                                     sigma=1.5, use_sample_covariance=False, full=True)
                maps.append(float(spatial[support].mean()))
        result[f"{name}_ssim"] = float(np.mean(maps)) if maps else None
        if name != "nir":
            p, t = p[:, mask], t[:, mask]
            pn, tn = np.linalg.norm(p, axis=0), np.linalg.norm(t, axis=0)
            cosine = np.divide((p * t).sum(0), pn * tn, out=np.zeros_like(pn), where=(pn * tn) > 0)
            cosine[(pn == 0) & (tn == 0)] = 1
            result[f"{name}_sam_degrees"] = float(np.degrees(np.arccos(np.clip(cosine, -1, 1))).mean())
    result["ssim_valid_fraction"] = float(support.mean())
    return result

In [ ]:
def infer(tensor, variant):
    return model(tensor, sampler=variant["sampler"], sampling_steps=variant["steps"],
                 sampling_eta=variant["eta"], sampling_temperature=1.0,
                 spectral_correction=SPECTRAL_CORRECTION, verbose=False)

measurements = []
preview_predictions = {}
measurement_path = RUN_DIR / "sampler_observations.jsonl"
# Re-running this cell would mix observations from separate timing sessions.
if measurement_path.exists():
    raise RuntimeError("This run already has measurements. Rerun configuration for a fresh directory.")
status_path = RUN_DIR / "status.json"
status_path.write_text(json.dumps({"status": "running"}))
nfe = [0]
def count_denoiser(module, inputs):
    nfe[0] += 1
hook = model.model.model.diffusion_model.register_forward_pre_hook(count_denoiser)
try:
    with torch.inference_mode():
        tensor = torch.from_numpy(pairs[0][0]).unsqueeze(0).to(device)
        for variant in tqdm(variants, desc="Untimed warmups"):
            for _ in range(WARMUPS_PER_VARIANT):
                seed_all(SEEDS[0])
                warmup = infer(tensor, variant)
                synchronize()
                del warmup
        progress = tqdm(total=len(pairs) * len(SEEDS) * len(variants), desc="Timed predictions")
        try:
            for image_index, (lr, hr, mask, meta) in enumerate(pairs):
                tensor = torch.from_numpy(lr).unsqueeze(0).to(device)
                for seed in SEEDS:
                    order = list(variants)
                    random.Random(seed + image_index).shuffle(order)
                    for position, variant in enumerate(order):
                        seed_all(seed)
                        nfe[0] = 0
                        synchronize()
                        if device.type == "cuda":
                            torch.cuda.reset_peak_memory_stats(device)
                        start = time.perf_counter()
                        prediction = infer(tensor, variant)
                        synchronize()
                        seconds = time.perf_counter() - start
                        if nfe[0] != variant["steps"]:
                            raise RuntimeError(f"Observed {nfe[0]} evaluations for {variant}.")
                        peak = torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None
                        array = prediction.squeeze(0).float().cpu().numpy()
                        del prediction
                        metrics = image_metrics(array, hr, mask)
                        row = dict(image_index=image_index, sample_id=selected_pairs[image_index]["sample_id"],
                                   seed=seed, **variant, nfe=nfe[0], seconds=seconds,
                                   execution_order=position, peak_cuda_allocated_bytes=peak,
                                   spectral_correction=SPECTRAL_CORRECTION, **meta,
                                   **{f"hr_{k}": v for k, v in metrics.items()})
                        with measurement_path.open("a") as stream:
                            stream.write(json.dumps(row, allow_nan=False) + "\n")
                        measurements.append(row)
                        if image_index == 0 and seed == SEEDS[0]:
                            preview_predictions[variant["variant"]] = array.copy()
                        progress.update(1)
        finally:
            progress.close()
    status_path.write_text(json.dumps({"status": "complete", "observations": len(measurements)}))
except BaseException as exc:
    status_path.write_text(json.dumps({"status": "failed", "error": str(exc),
                                      "observations": len(measurements)}))
    raise
finally:
    hook.remove()
np.savez_compressed(RUN_DIR / "preview_predictions.npz", **preview_predictions)
print("Saved individual observations:", measurement_path)

## 6. Paired summaries and confidence intervals

Each speedup is the matched `DDIM-100 seconds / candidate seconds`, averaged
across seeds within each image and then across images. It is **not the ratio of
aggregate mean runtimes**. Quality changes are paired against the same baseline.
95% percentile bootstrap intervals resample spatial images after averaging seeds.
With one image, intervals are omitted because image-to-image uncertainty cannot be estimated.

In [ ]:
def paired_summary(observations):
    frame = pd.DataFrame(observations)
    if frame.duplicated(["image_index", "seed", "variant"]).any():
        raise ValueError("Duplicate measurements.")
    expected = {(i, s, v["variant"]) for i in range(NUM_IMAGES)
                for s in SEEDS for v in variants}
    if set(zip(frame.image_index, frame.seed, frame.variant)) != expected:
        raise ValueError("Incomplete run: report only complete matched experiments.")
    baseline = frame.loc[frame.variant == "ddim_s100",
                         ["image_index", "seed", "seconds", "hr_rgb_ssim", "hr_nir_ssim",
                          "hr_rgb_psnr", "hr_nir_psnr"]]
    frame = frame.merge(baseline, on=["image_index", "seed"], suffixes=("", "_baseline"),
                        validate="many_to_one")
    frame["paired_speedup"] = frame.seconds_baseline / frame.seconds
    for band in ("rgb", "nir"):
        for metric in ("ssim", "psnr"):
            key = f"{band}_{metric}"
            frame[f"delta_{key}"] = frame[f"hr_{key}"] - frame[f"hr_{key}_baseline"]
    keys = ["seconds", "paired_speedup", "hr_rgb_ssim", "hr_rgb_psnr",
            "hr_nir_ssim", "hr_nir_psnr", "delta_rgb_ssim", "delta_rgb_psnr",
            "delta_nir_ssim", "delta_nir_psnr"]
    rows = []
    for variant in variants:
        group = frame.loc[frame.variant == variant["variant"]]
        # Seeds are repeated observations of an image, not independent samples.
        per_image = group.groupby("image_index", sort=True)[keys].mean()
        row = {**variant, "images": len(per_image), "observations": len(group)}
        for key in keys:
            values = per_image[key].to_numpy()
            if not np.isfinite(values).all():
                row[key] = row[f"{key}_ci_low"] = row[f"{key}_ci_high"] = None
                continue
            row[key] = float(values.mean())
            if len(values) > 1:
                rng = np.random.default_rng(2026)
                draws = rng.choice(values, (BOOTSTRAP_RESAMPLES, len(values)), replace=True).mean(1)
                low, high = np.percentile(draws, [2.5, 97.5])
                row[f"{key}_ci_low"], row[f"{key}_ci_high"] = float(low), float(high)
            else:
                row[f"{key}_ci_low"] = row[f"{key}_ci_high"] = None
        rows.append(row)
    return frame, pd.DataFrame(rows)

observations, summary = paired_summary(measurements)
observations.to_csv(RUN_DIR / "sampler_observations.csv", index=False)
summary.to_csv(RUN_DIR / "sampler_measurements.csv", index=False)
display(summary[["variant", "steps", "images", "seconds", "paired_speedup",
                 "hr_rgb_ssim", "delta_rgb_ssim", "hr_nir_ssim", "hr_rgb_psnr"]].round(4))

## 7. Speed and quality plots

Higher reference SSIM/PSNR measures agreement with the paired HR image; it does
not establish better perceptual detail. Cross-sensor and date differences also
affect these metrics. Use the visual comparison to inspect actual outputs.

In [ ]:
COLORS = {"ddim": "#2468b4", "dpm_solver++": "#19805c"}
LABELS = {"ddim": "DDIM (eta 0.95)", "dpm_solver++": "DPM-Solver++ 2M"}

def plot_curve(axis, xkey, ykey, intervals=True):
    for sampler, group in summary.groupby("sampler", sort=False):
        group = group.sort_values("steps")
        x, y = group[xkey].to_numpy(), group[ykey].to_numpy(dtype=float)
        axis.plot(x, y, "o-", color=COLORS[sampler], label=LABELS[sampler])
        for _, row in group.iterrows():
            axis.annotate(str(int(row.steps)), (row[xkey], row[ykey]),
                          xytext=(4, 5), textcoords="offset points", fontsize=8)
        if intervals and xkey == "steps":
            lo, hi = group[f"{ykey}_ci_low"], group[f"{ykey}_ci_high"]
            if lo.notna().all() and hi.notna().all():
                axis.fill_between(x, lo.to_numpy(dtype=float), hi.to_numpy(dtype=float),
                                  color=COLORS[sampler], alpha=0.12)
    axis.grid(alpha=0.2)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
plot_curve(axes[0], "steps", "paired_speedup")
axes[0].set(xscale="log", xlabel="Denoiser evaluations", ylabel="Paired speedup vs DDIM-100")
axes[0].axhline(1, color="gray", linestyle=":")
plot_curve(axes[1], "paired_speedup", "hr_rgb_ssim", intervals=False)
axes[1].set(xlabel="Paired speedup vs DDIM-100", ylabel="RGB SSIM vs HR")
axes[0].legend()
fig.suptitle(f"v1 · {NUM_IMAGES} images × {len(SEEDS)} seeds · {SPECTRAL_CORRECTION} · "
             + (provenance["gpu"] or "CPU"))
fig.savefig(RUN_DIR / "sampler_speed_quality.png", dpi=180)
plt.show()

fig, axes = plt.subplots(3, 2, figsize=(13, 12), constrained_layout=True)
for axis, key, label in zip(axes.flat,
        ["seconds", "paired_speedup", "hr_rgb_psnr", "hr_nir_psnr", "delta_rgb_ssim", "delta_nir_ssim"],
        ["Full forward time (s)", "Paired speedup vs DDIM-100", "RGB PSNR (dB)", "NIR PSNR (dB)",
         "RGB SSIM change vs DDIM-100", "NIR SSIM change vs DDIM-100"]):
    plot_curve(axis, "steps", key)
    axis.set(xscale="log", xlabel="Denoiser evaluations", ylabel=label)
    if key.startswith("delta_"):
        axis.axhline(0, color="gray", linestyle=":")
axes[0, 0].legend()
fig.suptitle("95% confidence intervals bootstrap spatial images after averaging seeds")
fig.savefig(RUN_DIR / "sampler_diagnostics.png", dpi=180)
plt.show()

In [ ]:
# First fixed image and first requested seed; no selection by apparent quality.
lr, hr = pairs[0][:2]
low, high = np.percentile(lr[:3], [2, 98])
def display_rgb(array):
    return np.clip((array[:3].transpose(1, 2, 0) - low) / max(high - low, 1e-6), 0, 1)

panels = [("Sentinel-2 LR", lr), ("HR reference", hr)]
for name, label in [("ddim_s100", "DDIM · 100 steps"), ("dpm_2m_s1", "DPM++ · 1 step"),
                    ("dpm_2m_s3", "DPM++ · 3 steps"), ("dpm_2m_s5", "DPM++ · 5 steps")]:
    if name in preview_predictions:
        panels.append((label, preview_predictions[name]))
fig, axes = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4.5), constrained_layout=True)
for axis, (label, image) in zip(np.atleast_1d(axes), panels):
    axis.imshow(display_rgb(image), interpolation="nearest")
    axis.set_title(label)
    axis.axis("off")
fig.suptitle(f"{selected_pairs[0]['sample_id']} · seed {SEEDS[0]} · shared LR RGB stretch")
fig.savefig(RUN_DIR / "sampler_comparison.png", dpi=180)
plt.show()

# Export only this run's results, not cached datasets or checkpoint weights.
import zipfile
archive_path = RUN_DIR.with_suffix(".zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.iterdir()):
        if path.is_file():
            archive.write(path, path.name)
print("Results:", RUN_DIR)
print("Downloadable archive:", archive_path)
try:
    from google.colab import files as colab_files
except ImportError:
    from IPython.display import FileLink
    display(FileLink(str(archive_path)))
else:
    colab_files.download(str(archive_path))

The individual observations and provenance remain available if you want to
replot or compare runs. Keep the hardware, source revision, cohort, precision,
seeds and correction mode with any reported speedup. The summary CSV includes
the columns used by `resources/sampler_info/sampler_measurements.csv` plus NIR
metrics and their confidence intervals. A failed or interrupted run retains its
JSONL observations and failure status; start a new run before generating a report.